# Labeling Script also Providing Defnitions and Examples  

## TOC 

- [Setup](#setup)
- [Load, Partition, and Sample Tweets/Batches for Labeling](#load_tweets)
- [Define the different prompts](#define_prompts)
- [Run the Labeling Tasks (Single)](#run_labeling_single)
- [Run the Labeling Tasks (Single) + Confidence](#run_labeling_single_conf)
- [Run the Labeling Tasks (Batch)](#run_labeling_batch)
- [Run the Labeling Tasks (Batch) + Confidence](#run_labeling_batch_confidence)

#### Open ToDo's
- Clean tweets from nbsps etc.  
- rewrite data frame creation and data collection into separate functions  

- Update Collection prompt s.t. there are now unneccessary ' after splitting
 --> batch conf still contains some cases without proper scores

Update to previous versions: 

-separation of task by system message (prompt and def) and user message (tweet)

In [101]:
# General Setup and Git
# cd to path of current script
import os
import sys
os.chdir(sys.path[0])

! git pull

Already up to date.


In [ ]:
# Git codes
## see git status
! git status # see git status

# Ask the user for a commit message
commit_message = input("Enter commit message: ")

## Git add the current stuff to commit and set commit message
! git add .
! git commit -m "$commit_message"

## Push commits to remote
! git push

On branch master
Your branch is up to date with 'origin/master'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .ipynb_checkpoints/4c_GPT_Labeling_Definition_FewShot_refined-checkpoint.ipynb
	modified:   .jupyter/desktop-workspaces/default-37a8.jupyterlab-workspace
	modified:   4b_GPT_Labeling_Definition_FewShot.ipynb
	modified:   4c_GPT_Labeling_Definition_FewShot_refined.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


<a id="setup"></a>
## SANDBOX & SETUP

In [ ]:
# Load API key from gitignored api_key module 
from utils.api_key import API_KEY as API_KEY

In [3]:
# base Request
# test capabilties
from openai import OpenAI

# Setup open AI client using api key
client = OpenAI(
  api_key=API_KEY
)

# Test prompt
return_number = "Return me a number between 1 and 100. Only return the number"

# Test completion
test_response = client.chat.completions.create(
  model="gpt-4o-mini",
  store=True,
  messages=[
    {"role": "user", 
     "content": return_number
    }
  ],
  temperature = 1 # default temp = 1  
)

print("Response: ", test_response.choices[0].message.content);

Response:  42


<a id="load_tweets"></a>
## Load, Partition, and Sample Tweets for Labeling

### Load the Tweets from the Kern et al. (2023) paper

In [4]:
# load the full tweets as used by CK
import pandas as pd
# pd.set_option('max_colwidth', 1000)
pd.set_option('display.max_colwidth', 0)

# Load the tweet files
folder_path = 'Tweets_CK/'
train_file = 'full_train_s.csv'
test_file = 'full_test_s.csv'

train_df = pd.read_csv(os.path.join(folder_path, train_file))
test_df = pd.read_csv(os.path.join(folder_path, test_file))

# Reorder test_df columns to match train_df
test_df = test_df[train_df.columns]

# Add the 'original_split' column
train_df['original_split'] = 'train'
test_df['original_split'] = 'test'

# Combine them
tweets_full = pd.concat([train_df, test_df], ignore_index=True)

# Rename columns for a consistent style 
tweets_full = (tweets_full.
               rename(columns={'tweet.id': 'tweet_id', 
                               'batch.tweet': 'batch_tweet',
                               'hate.speech': 'hate_speech', 
                               'offensive.language': 'offensive_language'}))

In [5]:
# Get Basic Info 
num_rows = tweets_full.shape[0]
num_cols = tweets_full.shape[1]
num_unique_tweets = tweets_full['tweet_id'].nunique()
tweet_counts = (tweets_full[['tweet_hashed','batch_tweet']]
                .drop_duplicates().groupby('batch_tweet').size())
num_batches = tweet_counts.mean()
average_tweets_per_batch = tweets_full['batch_tweet'].nunique()

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Number of unique tweet IDs: {num_unique_tweets}")
print(f"Number of Batches: {num_batches}")
print(f'Average Tweets per Batch: {average_tweets_per_batch}')
tweets_full.sort_values('tweet_id').head()

Number of rows: 44900
Number of columns: 8
Number of unique tweet IDs: 3000
Number of Batches: 60.0
Average Tweets per Batch: 50


,tweet_id,id,version,batch_tweet,hate_speech,offensive_language,tweet_hashed,original_split
2927,1,7,B,R1,0.0,1.0,@###### bro that hoe live,train
2930,1,13,A,R1,0.0,1.0,@###### bro that hoe live,train
2932,1,155,A,R1,0.0,1.0,@###### bro that hoe live,train
2918,1,125,E,R1,1.0,0.0,@###### bro that hoe live,train
2919,1,173,E,R1,0.0,1.0,@###### bro that hoe live,train


### Partition the tweets into new Batches with IDs
In the original papers, the batch creation process does not seem to be included in the code.  
However, the tweets were randomly grouped into batches of 50.  
It is unclear whether this grouping was done based on the original tweet id gathered from elsewhere or whether the IDs were enumerated afterwards.  
In the end, we have have 50 tweets per batch (60 batches accordingly) that always appeared in the same order per batch.  
This randomized batch allocation process is replicated in the following.

In [69]:
# Randomly partition the df into batches reorder the tweets within batches 
BATCH_SIZE = 6

seed = 13
tweets_full_shuffled = (tweets_full[['tweet_id','tweet_hashed']]
                        .drop_duplicates()
                        .sample(frac=1, random_state=seed)
                        .reset_index(drop=True))
# create batch IDs
tweets_full_shuffled['batch_id'] = (tweets_full_shuffled.index // BATCH_SIZE) + 1
# create position IDs for each tweet within its batch
tweets_full_shuffled['tweet_in_batch'] = (tweets_full_shuffled.index % BATCH_SIZE) + 1

tweets_full_shuffled.head(5)

,tweet_id,tweet_hashed,batch_id,tweet_in_batch
0,811,Fat fucking funky nasty ass hoes,1,1
1,742,Up early then a bitch driving to denton omg can I move already,1,2
2,2098,@###### @###### there isn't a green one either. There's red and yellow in that pic...,1,3
3,2349,The only thing about niccas is they be followers smh .. Be ya self everyone else is taken!!,1,4
4,1371,@###### I'm the type to put that bitch in a river so you can gtf out my locations lol,1,5


### Clean tweets from escape characters etc.

In [70]:
# TO DO?

### Randomly Sample N tweets via batches and create the result df out of them

In [71]:
# sample tweets using the batch_ids 
# i.e., sample count = n_batch_samples * batch size 
# N_BATCH_SAMPLES = 20
N_BATCH_SAMPLES = 2
sampled_batches = (pd.Series(tweets_full_shuffled['batch_id']
                   .unique())
                   .sample(n=N_BATCH_SAMPLES
                           , random_state=13
                          ))

# get the tweets in these batches 
sampled_tweets = (tweets_full_shuffled[tweets_full_shuffled['batch_id']
                                       .isin(sampled_batches)]
                  [['batch_id','tweet_in_batch', 'tweet_id','tweet_hashed']]
                  .drop_duplicates())

print(f'Number of Batches Sampled: {N_BATCH_SAMPLES}')
print(f"Number of Unique Tweets Sampled: {len(sampled_tweets['tweet_hashed'])}")
print(f'Sampled Batch IDs: {sampled_batches.to_list()}')
sampled_tweets.iloc[0:3]

Number of Batches Sampled: 2
Number of Unique Tweets Sampled: 12
Sampled Batch IDs: [1, 335]


,batch_id,tweet_in_batch,tweet_id,tweet_hashed
0,1,1,811,Fat fucking funky nasty ass hoes
1,1,2,742,Up early then a bitch driving to denton omg can I move already
2,1,3,2098,@###### @###### there isn't a green one either. There's red and yellow in that pic...


<a id="define_prompts"></a>
## Define the different Prompts

In [64]:
def_and_examples = """
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not hate speech. 
    For example,  
    “Why no boycott of the racist “redskins”? #Redskins @ChangeTheName”  
    This tweet contains a hateful term, but the tweeter is making an anti-racist statement. We would label tweet as containing offensive language.

    We define Offensive Language as:  
    Language that is highly offensive to certain individuals or groups but does not meet the requirements of hate speech. 

    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    This tweet contains offensive language, but it is not hate speech.   
    We would label it as offensive language.  

    Some tweets contains both hate speech AND offensive language.  
    “Subtweet me one more time, you dirty chink whore”  

    Some tweets do not meet our definition of hate speech or offensive language.  
    “Great lead battle and then Ricky hits Danica for a yellow. Oh boy. #NASCAR”  
"""



# Define the prompts
PROMPT_DICT = {
    # SINGLE TWEET PROMPTING
    ### A: Both OL and HS at the same time, OL first
    "A":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    {explanation}
    
    For the following tweets, your task is to: 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL
""", 
    ### B: Both OL and HS at the same time, HS first
    "B":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or Offensive Language (OL). 
    {explanation}
    
    For the following tweets, your task is to: 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL
    """, 
    ### C.OL: Only OL
    "C.OL":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    {explanation}
    
    For the following tweets, your task is to: 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL
""", 
    ### C.HS: Only HS
    "C.HS":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    {explanation}
    
    For the following tweets, your task is to: 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    
    Example Input: 
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Example Output:
    HS
""",
    
    
    # SINGLE TWEET PROMPTING + CONFIDENCE SCORE
    ### A: Both OL and HS at the same time, OL first
    "A_conf":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    {explanation}
    
    For the following tweets, your task is to: 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL, 4
""", 
    ### B: Both OL and HS at the same time, HS first
    "B_conf":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or Offensive Language (OL). 
    {explanation}
    
    For the following tweets, your task is to: 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL, 4
""", 
    ### C.OL: Only OL
    "C.OL_conf":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    {explanation}
    
    For the following tweets, your task is to: 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL, 4 
""", 
    ### C.HS: Only HS
    "C.HS_conf":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    {explanation}
    
    For the following tweets, your task is to: 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    
    Example Input: 
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Example Output:
    HS, 4
""",
    
    
   

    # BATCH PROMPTING (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch":
    """You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL) or Hate Speech (HS). 
    {explanation}
    
    For the following tweets, your task is to: 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    For each tweet return only one label.
    Separate the the chosen labels for the different tweets with ';'
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  
    
    Example Output:
    OL, 4; OLHS, 4
    
""", 
    ### B: Both OL and HS at the same time, HS first
    "B_batch":
    """You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS) or Offensive Language (OL). 
    {explanation}
    
    For the following tweets, your task is to: 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  
    
    Example Output:
    OL; HSOL, 4
""", 
    ### C.OL: Only OL
    "C.OL_batch":
    """You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL). 
    {explanation}
    
    For the following tweets, your task is to: 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  
    
    Example Output:
    OL, 4; OL, 4
""", 
    ### C.HS: Only HS
    "C.HS_batch":
    """You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS). 
    {explanation}
    
    For the following tweets, your task is to: 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  
    
    Example Output:
    none, 4; OL, 4
""",
    
    
    # BATCH PROMPTING WITH CONFIDENCE SCORES (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch_conf":
    """You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL) or Hate Speech (HS). 
    {explanation}
    
    For the following tweets, your task is to: 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    For each tweet,only return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Separate label and confidence score by ','  
    Separate the responses for the different tweets by ';'
    Do not add any further explanations.
    Important: Ensure to return exactly one label and one score per tweet provided.  
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  
    
    Example Output:
    OL, 4; OLHS, 4
""", 
    ### B: Both OL and HS at the same time, HS first
    "B_batch_conf":
    """You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS) or Offensive Language (OL). 
    {explanation}
    
    For the following tweets, your task is to: 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    For each tweet,only return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Separate label and confidence score by ','  
    Separate the responses for the different tweets by ';'
    Do not add any further explanations.
    Important: Ensure to return exactly one label and one score per tweet provided.   
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  
    
    Example Output:
    OL; HSOL, 4
""", 
    ### C.OL: Only OL
    # Each tweet starts in a new line and is numbered with its index number and 3 closed brackets.
    "C.OL_batch_conf":
    """You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL). 
    {explanation}
    
    For the following tweets, your task is to: 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    For each tweet,only return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Separate label and confidence score by ','  
    Separate the responses for the different tweets by ';'
    Do not add any further explanations.
    Important: Ensure to return exactly one label and one score per tweet provided.  
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  
    
    Example Output:
    OL, 4; OL, 4
""", 
    ### C.HS: Only HS
    "C.HS_batch_conf":
    """You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS). 
    {explanation}
    
    For the following tweets, your task is to: 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    For each tweet,only return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Separate label and confidence score by ','  
    Separate the responses for the different tweets by ';'
    Do not add any further explanations.
    Important: Ensure to return exactly one label and one score per tweet provided.  
    
    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  
    
    Example Output:
    none, 4; OL, 4
"""
}

<a id="run_labeling_single"></a>
## Run the Labeling Tasks (Single Tweet Prompting)

### Create Result DF

In [65]:
# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of the ones in the prompt dictionary)

CONDITIONS = ['A', 'B', 'C.HS', 'C.OL']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,1,1,811,Fat fucking funky nasty ass hoes,A,,,
1,1,1,811,Fat fucking funky nasty ass hoes,B,,,
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS,,,
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL,,,
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A,,,


In [55]:
%%time 

# Run the actual Loop for API Calls 

for row in range(0,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt_task = (PROMPT_DICT[condition]
          .format(explanation = def_and_examples))
    
    # Define message
    messages = [{"role": "developer", "content": prompt_task},
                {"role": "user",   "content": tweet_to_rate}]
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        # get response via API Call
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages= messages,
            temperature=1,
            store=True,
        )
        label = resp.choices[0].message.content.strip()
        # print(prompt)
        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"R{response}_label"] = label
        
    # Print Progress
    print(f"""Finished Row {row+1} of {results.shape[0]} --> {round((row+1)/results.shape[0]*100,1)} %""", 
          end='\r')

n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results

Number of API Calls: 120> 100.0 %
CPU times: user 1.54 s, sys: 100 ms, total: 1.64 s
Wall time: 1min 16s


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,115,1,2407,"@###### @###### @###### They have mix, not all are retards. AAP is Maoists, terrorists,Congies and delusional Retards",A,OLHS,OLHS,OLHS
1,115,1,2407,"@###### @###### @###### They have mix, not all are retards. AAP is Maoists, terrorists,Congies and delusional Retards",B,HSOL,HSOL,HSOL
2,115,1,2407,"@###### @###### @###### They have mix, not all are retards. AAP is Maoists, terrorists,Congies and delusional Retards",C.HS,HS,HS,HS
3,115,1,2407,"@###### @###### @###### They have mix, not all are retards. AAP is Maoists, terrorists,Congies and delusional Retards",C.OL,OL,OL,OL
4,115,2,2631,@###### Dumb Haitian fake black faggots. Go to Haiti and neck yourself.,A,OLHS,OLHS,OLHS
5,115,2,2631,@###### Dumb Haitian fake black faggots. Go to Haiti and neck yourself.,B,HS,HS,HS
6,115,2,2631,@###### Dumb Haitian fake black faggots. Go to Haiti and neck yourself.,C.HS,HS,HS,HS
7,115,2,2631,@###### Dumb Haitian fake black faggots. Go to Haiti and neck yourself.,C.OL,OL,OL,OL
8,115,3,1482,you drove me redneck crazy,A,none,OL,OL
9,115,3,1482,you drove me redneck crazy,B,OL,OL,OL


In [18]:
# save result file 
import os 
from datetime import date

tweet_count = (results['tweet'].drop_duplicates()
               .shape[0])
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confno"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_single_conf"></a>
## Run the Labeling Task (with confidence scores)

### Create Result Df

In [79]:
# ISSUE: Loop is executed sequential s.t. we cannot use the acctual query-per-sec quote of the API (3r/s)
# SOL: --> rewrite code s.t. it's parallelized

# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_conf', 'B_conf', 'C.OL_conf', 'C.HS_conf']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_score"] = ""    
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_Token1"] = ""   
    results[f"R{i}_Token2"] = ""  
    results[f"R{i}_Token3"] = ""      

    
results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score,R1_Token1,R1_Token2,R1_Token3,R2_Token1,R2_Token2,R2_Token3,R3_Token1,R3_Token2,R3_Token3
0,1,1,811,Fat fucking funky nasty ass hoes,A_conf,,,,,,,,,,,,,,,
1,1,1,811,Fat fucking funky nasty ass hoes,B_conf,,,,,,,,,,,,,,,
2,1,1,811,Fat fucking funky nasty ass hoes,C.OL_conf,,,,,,,,,,,,,,,
3,1,1,811,Fat fucking funky nasty ass hoes,C.HS_conf,,,,,,,,,,,,,,,
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_conf,,,,,,,,,,,,,,,


### Run the Loop and score confidence scores

In [57]:
row = 1

# Get Tweets and Conditions for the Loop
tweet_to_rate = results.iloc[row]['tweet']
condition = 'A_conf'
prompt = (PROMPT_DICT[condition]
          .format(tweet=tweet_to_rate, explanation = def_and_examples))

print(prompt)


resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=1,
    store=True,
)
label = resp.choices[0].message.content.strip()
print(f'Response: {label}')

You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not hate

In [80]:
# %%time 

# Run the actual Loop for API Calls 
N_TOP_LOGPROBS = 5

for row in range(0,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt_task = (PROMPT_DICT[condition]
               .format(explanation = def_and_examples))
    
    # Define message
    messages = [{"role": "developer", "content": prompt_task},
                {"role": "user",   "content": tweet_to_rate}]
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        # get response via API Call
        resp_full = client.chat.completions.create(
            model="gpt-4o-mini",
            messages= messages,
            temperature=1,
            logprobs=True,
            top_logprobs= N_TOP_LOGPROBS,
            store=True,
        )
        # Extract the Labels & Confidence Score
        resp = resp_full.choices[0].message.content.strip()
        resp_split = resp.split(',')
        label = resp_split[0]
        score = resp_split[1].strip()
        
        # Extract the Token Logprobs for tokens 1 to 3
        # exract logprobs for first token
        top_k_logprobs_first = resp_full.choices[0].logprobs.content[0].top_logprobs
        top_n_tokens_first = [obj.token for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        top_n_logprobs_first = [obj.logprob for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        # token_value_dict = dict(zip(top_n_tokens_first, top_n_logprobs_first))
        token_value_series_first = pd.Series(top_n_logprobs_first, index=top_n_tokens_first)

        # exract logprobs for second token
        top_k_logprobs_second = resp_full.choices[0].logprobs.content[1].top_logprobs
        top_n_tokens_second = [obj.token for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        top_n_logprobs_second = [obj.logprob for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        token_value_series_second = pd.Series(top_n_logprobs_second, index=top_n_tokens_second)

        # exract logprobs for third token
        top_k_logprobs_third = resp_full.choices[0].logprobs.content[2].top_logprobs
        top_n_tokens_third = [obj.token for obj in top_k_logprobs_third[:N_TOP_LOGPROBS]]
        top_n_logprobs_third = [obj.logprob for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        token_value_series_third = pd.Series(top_n_logprobs_third, index=top_n_tokens_third)

        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"R{response}_label"] = label
        results.at[row, f"R{response}_score"] = score
        results.at[row, f"R{response}_Token1"] = token_value_series_first
        results.at[row, f"R{response}_Token2"] = token_value_series_second
        results.at[row, f"R{response}_Token3"] = token_value_series_third
        
    # Print Progress
    print(f"""Finished Row {row+1} of {results.shape[0]} --> {round((row+1)/results.shape[0]*100,1)} %""", 
          end='\r')


n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(4)

KeyboardInterrupt: 

In [23]:
# save result file 
import os 
from datetime import date

tweet_count = (results['tweet'].drop_duplicates()
               .shape[0])
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confyes"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_batch"></a>
## Run the Labeling Tasks (Batch Prompting)

### Create Results DF

In [82]:
# SELECT CONDITIONS TO RUN THE LABELING FOR

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_batch', 'B_batch', 'C.HS_batch', 'C.OL_batch']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)
results = results.sort_values(['batch_id', 'tweet_in_batch', 'condition'])

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""

results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch,,,
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch,,,
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch,,,
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch,,,
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch,,,


### Run the Loop

In [89]:
%%time 

# Run the actual Loop for API Calls 
# now we have too loop through conditions first and batches second
# now we have to loop through batches and provide all tweets in the prompt 
# then we will get different responses 

conditions = results['condition'].unique()
batches = results['batch_id'].unique()

N_RESPONSES = 3

# one condition after the other
for condition in conditions:
    # print(condition)
    
    # loop through batches in the conditions
    for batchID in batches:
        
        
        # get the tweets within each batch and create the prompt 
        tweets_in_batch = (results[results['batch_id'] == batchID]
                       ['tweet']
                       .unique())
        tweet_indices = ((results['batch_id'] == batchID) & 
                         (results['condition'] == condition))
        tweets_in_batch_str = "\n" + "\n".join(
            [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
        )
        prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        
        # Define message
        messages = [{"role": "developer", "content": prompt_task},
                    {"role": "user",   "content": tweets_in_batch_str}]
        
        
        
        
        # generate N_RESPONSES responses
        for response in range(1, N_RESPONSES + 1):
            # get response via API Call
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=1,
                store=True,
            )
            labels = resp.choices[0].message.content.strip()
            labels_list = labels.split(';')
            labels_list

        
            # Store the different labels in the correct df rows 
            ### Get the result df indices for the tweets in the current batch
            subset_indices = results.loc[tweet_indices].index.to_list()
            # store results
            for i, idx in enumerate(subset_indices):
                results.at[idx, f"R{response}_label"] = labels_list[i]
                

        # Print Progress
        message = f"""Finished Batch {batches.tolist().index(batchID) + 1} in Condition {conditions.tolist().index(condition) + 1} --> \
        {((conditions.tolist().index(condition)) * len(batches) + (batches.tolist().index(batchID) + 1)) / (len(conditions) * len(batches)) * 100:.2f} %"""
        print('\r' + message.ljust(100), end='')
        sys.stdout.flush()        
       
    
n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(5)

Finished Batch 2 in Condition 4 -->         100.00 %                                                Number of API Calls: 144
CPU times: user 287 ms, sys: 23.2 ms, total: 310 ms
Wall time: 24.7 s


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch,OL,OL,OL
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch,OL,OL,OL
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch,none,none,none
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch,OL,OL,OL
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch,OL,OL,OL


In [27]:
# save result file 
import os 
from datetime import date

tweet_count = (results['tweet'].drop_duplicates()
               .shape[0])
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confno_batch"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_batch_confidence"></a>
## Run the Labeling Tasks (Batch Prompting) + Self-Report Confidence Scores

### Create Result Df (use the sample sampled batches as before)

In [95]:
# SELECT CONDITIONS TO RUN THE LABELING FOR

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_batch_conf', 'B_batch_conf', 
              'C.HS_batch_conf', 'C.OL_batch_conf']

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)
results = results.sort_values(['batch_id', 'tweet_in_batch', 'condition'])

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_score"] = ""    

results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch_conf,,,,,,
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch_conf,,,,,,
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch_conf,,,,,,
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch_conf,,,,,,
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch_conf,,,,,,


### Run the Loop

In [99]:
# testrun to check output format

batchID = 1
condition = 'A_batch_conf'

# batchID = results['batch_id'][0]
# condition = results['condition'][0]
tweets_in_batch = (results[results['batch_id'] == batchID]
                   ['tweet']
                   .unique())
tweets_in_batch_str = "\n" + "\n".join(
    [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
)

tweets_in_batch_str_test = 'Before each label, return the index number of the respective tweet.' + tweets_in_batch_str


prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        
# Define message
messages = [{"role": "developer", "content": prompt_task},
            {"role": "user",   "content": tweets_in_batch_str}]

resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=1,
            store=True,
        )

response = resp.choices[0].message.content.strip()
responses = response.split(';')
responses

print(prompt_task)
print(tweets_in_batch_str)

print(f"\n Response: ", response)
# labels = resp.choices[0].message.content.strip()
# labels_list = labels.split(';')
# labels_list

You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL) or Hate Speech (HS). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms,

In [100]:
# %%time 

# Run the actual Loop for API Calls 
# now we have too loop through conditions first and batches second
# now we have to loop through batches and provide all tweets in the prompt 
# then we will get different responses 

conditions = results['condition'].unique()
batches = results['batch_id'].unique()

N_RESPONSES = 3

# one condition after the other
for condition in conditions:
    # print(condition)
    
    # loop through batches in the conditions
    for batchID in batches:
        
        # get the tweets within each batch and create the prompt 
        tweets_in_batch = (results[results['batch_id'] == batchID]
                       ['tweet']
                       .unique())
        tweet_indices = ((results['batch_id'] == batchID) & 
                         (results['condition'] == condition))
        tweets_in_batch_str = "\n" + "\n".join(
            [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
        )
        prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        messages = [{"role": "developer", "content": prompt_task},
                    {"role": "user",   "content": tweets_in_batch_str}]

        
        # generate N_RESPONSES responses
        for response in range(1, N_RESPONSES + 1):
            
            valid_response = False
            
            while not valid_response:
                
            
                # get response via API Call
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages= messages,
                    temperature=1,
                    store=True,
                )
                api_response_full = resp.choices[0].message.content.strip()
                api_responses = api_response_full.split(';')
                api_responses_per_tweet = [api_response.split(',') for api_response in api_responses]
                
                # Check if we have the desired format (10 responses and 10 commas produced)
                if len(api_responses_per_tweet) == 6:
                    valid_response = True
                else:
                    continue
                    

            # Extract the Labels & Confidence Scores per tweet
            labels_list = [tweet_response[0].strip() 
                           for tweet_response in api_responses_per_tweet]
            scores_list = [tweet_response[1].strip() 
                           for tweet_response in api_responses_per_tweet]


            # Store the different labels in the correct df rows 
            ### Get the result df indices for the tweets in the current batch
            subset_indices = results.loc[tweet_indices].index.to_list()
            # store results
            for i, idx in enumerate(subset_indices):
                results.at[idx, f"R{response}_label"] = labels_list[i]
                results.at[idx, f"R{response}_score"] = scores_list[i]
                
        # Print Progress
        message = f"""Finished Batch {batches.tolist().index(batchID) + 1} in Condition {conditions.tolist().index(condition) + 1} --> \
        {((conditions.tolist().index(condition)) * len(batches) + (batches.tolist().index(batchID) + 1)) / (len(conditions) * len(batches)) * 100:.2f} %"""
        print('\r' + message.ljust(100), end='')
        sys.stdout.flush()    
       
    
n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(5)

Finished Batch 2 in Condition 4 -->         100.00 %                                                Number of API Calls: 144


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch_conf,OLHS,OLHS,OLHS,4,4,4
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch_conf,OL,OL,OL,3,3,3
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch_conf,none,HS,none,4,3,4
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch_conf,OL,OL,OL,4,4,4
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch_conf,OL,OL,OL,3,3,4


In [32]:
# save result file 
import os 
from datetime import date

tweet_count = (results['tweet'].drop_duplicates()
               .shape[0])
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confyes_batch"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

# Notes 
1) The tweets can be processed in parts to either (e.g., the first 2k for all conditions)  
A) be able to split the task into free sub parts or   
B) to be able to better estimate the total costs  

2) Probably the Batch Calls have to be done in a separate code snippet

3) Costs: 
4o-mini : 
Input:          15ct  /1M Tokens
Cached Input:   7.5ct /1M Tokens (might be our case)
Output:         60ct  /1M Tokens


In [39]:
# Get Token Counts 
from utils import count_tokens
import statistics
# count tokens for gpt-3.5-turbo model (default in tiktoken package 
# print(PROMPT_DICT['A'])
# print(sampled_tweets['tweet_hashed'])
print(count_tokens(PROMPT_DICT['A'])) # 111 TOKENS for standard condition prompt 
print(statistics.fmean([count_tokens(x) for x in sampled_tweets['tweet_hashed']])) # 15 Average Tweet Token length
# response has max 2 tokens

121
21.94
